# Safebooru 메타데이터 크롤링
Safebooru API에서 메타데이터(태그 + URL)만 수집해 Parquet으로 저장합니다.
- 1000건마다 `1000.parquet`, `2000.parquet`, ... 형태로 순차 저장
- 중단 후 이어서 크롤링 가능 (기존 파일 자동 감지)
- **이미지는 저장하지 않음** — 학습 시 배치 단위로 임시 다운로드

In [ ]:
import pandas as pd
import os
import time
import requests
import urllib3
import ssl
import re
from requests.adapters import HTTPAdapter

# SSL 경고 무시
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class TLSAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context()
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        ctx.set_ciphers('DEFAULT@SECLEVEL=1')
        kwargs['ssl_context'] = ctx
        return super(TLSAdapter, self).init_poolmanager(*args, **kwargs)

# --- [경로 설정] ---
SAVE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "parquet"))
os.makedirs(SAVE_DIR, exist_ok=True)

# 인증 정보
USER_ID = "925305"
API_KEY = "b462a643571c0dd74c57131f82078b100dec94bd0bfd118fb0e7eb9b95b7182a"
BASE_URL = "https://gelbooru.com/index.php"

def get_last_pid():
    """저장된 폴더를 확인하여 마지막 페이지 번호(pid)를 반환"""
    files = [f for f in os.listdir(SAVE_DIR) if f.endswith('.parquet')]
    if not files:
        return -1  # 파일이 없으면 0번 pid부터 시작 (i+1=0이 되도록)
    
    # 파일명에서 숫자만 추출 (예: '100.parquet' -> 100)
    nums = []
    for f in files:
        match = re.search(r'(\+?\d+)', f)
        if match:
            nums.append(int(match.group(1)))
    
    if not nums:
        return -1
        
    # 가장 높은 숫자를 찾음 (예: 1000)
    last_num = max(nums)
    # 100단위 파일명이므로 100으로 나누면 마지막 pid가 나옴 (예: 1000 -> 10번째 페이지 수집완료)
    return (last_num // 100) - 1

def crawl_gelbooru_resume():
    session = requests.Session()
    session.mount('https://', TLSAdapter())
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
    })

    # 마지막 수집 위치 확인
    last_pid = get_last_pid()
    start_pid = last_pid + 1
    print(f"재개 지점 확인: { (start_pid + 1) * 100 }번 데이터부터 수집을 시작합니다.")

    # 충분히 큰 범위 설정 (원하는 만큼)
    for i in range(start_pid, start_pid + 500): 
        current_unit = (i + 1) * 100
        file_name = f"{current_unit}.parquet"
        file_path = os.path.join(SAVE_DIR, file_name)
        
        params = {
            'page': 'dapi',
            's': 'post',
            'q': 'index',
            'json': 1,
            'limit': 100,
            'pid': i,
            'tags': '', 
            'api_key': API_KEY,
            'user_id': USER_ID
        }
        
        try:
            response = session.get(BASE_URL, params=params, timeout=30, verify=False)
            
            if response.status_code == 200:
                res_data = response.json()
                posts = res_data.get('post', []) if isinstance(res_data, dict) else res_data
                
                if posts and len(posts) > 0:
                    df = pd.DataFrame(posts)
                    target_cols = ['id', 'tags', 'sample_url', 'file_url', 'width', 'height']
                    actual_cols = [c for c in target_cols if c in df.columns]
                    
                    df[actual_cols].to_parquet(file_path, engine='pyarrow', index=False)
                    print(f"[{current_unit}] 저장 성공")
                else:
                    print(f"[{current_unit}] 더 이상 데이터가 없습니다. 수집을 마칩니다.")
                    break
            else:
                print(f"[{current_unit}] 접속 실패 (HTTP {response.status_code})")
                break
                
        except Exception as e:
            print(f"[{current_unit}] 에러 발생: {e}")
            break
            
        time.sleep(1.0)

if __name__ == "__main__":
    crawl_gelbooru_resume()

저장 경로 확정: c:\Users\EL069\Project\safebooru\data\parquet
[100] 저장 성공: c:\Users\EL069\Project\safebooru\data\parquet\100.parquet
[200] 저장 성공: c:\Users\EL069\Project\safebooru\data\parquet\200.parquet
[300] 저장 성공: c:\Users\EL069\Project\safebooru\data\parquet\300.parquet
[400] 저장 성공: c:\Users\EL069\Project\safebooru\data\parquet\400.parquet
[500] 저장 성공: c:\Users\EL069\Project\safebooru\data\parquet\500.parquet
[600] 저장 성공: c:\Users\EL069\Project\safebooru\data\parquet\600.parquet
[700] 저장 성공: c:\Users\EL069\Project\safebooru\data\parquet\700.parquet
[800] 저장 성공: c:\Users\EL069\Project\safebooru\data\parquet\800.parquet
[900] 저장 성공: c:\Users\EL069\Project\safebooru\data\parquet\900.parquet
[1000] 저장 성공: c:\Users\EL069\Project\safebooru\data\parquet\1000.parquet
